Setup dell'ambiente (importiamo le librerie necessarie e facciamo i primi check):

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device usato:", device)

In un computer tutto è rappresentato numericamente, tramite _vettori_.

Un **vettore** è una lista di numeri (ovvero un elenco _ordinato_) in cui ogni numero (detto anche _entrata_ del vettore) è interpretato come una coordinata di un punto nello spazio.

Due (o più) vettori vivono nello stesso spazio se le coordinate sono interpretate allo stesso modo:

![vettori](images/vettori.png)

In python, le principali librerie con cui definire e lavorare con i vettori sono `numpy` e `pytorch` (quest'ultima permette anche l'uso della GPU).

In [ ]:
# un oggetto descritto da pochi numeri (es. le statistiche di un Pokémon)
stats = [45, 49, 49, 65, 45] # una lista Python

vect_numpy = np.array(stats) # lista -> array NumPy
vect_torch = torch.tensor(stats) # lista -> tensore PyTorch
torch_da_numpy = torch.from_numpy(vect_numpy) # array NumPy -> tensore
numpy_da_torch = vect_torch.numpy()

print(type(stats), stats)
print(type(vect_numpy), vect_numpy)
print(type(vect_torch), vect_torch)
print(type(torch_da_numpy), torch_da_numpy)
print(type(numpy_da_torch), numpy_da_torch)

Una lista di vettori diventa una _matrice_:

In [ ]:
batch = [[45, 49, 49, 65, 45],
         [39, 52, 43, 60, 65]]
Xb = torch.tensor(batch, dtype=torch.float32)
print(Xb)
print(Xb.shape) # (2, 5): 2 oggetti, ognuno un punto in 5 dimensioni

## Explorative Data Analysis

Un dataset è una tabella: ogni riga è un esempio (un dato, un oggetto), ogni colonna è una feature (una caratteristica misurata). `pandas` rappresenta questa tabella con un `DataFrame` (`df`).

L'Explorative Data Analysis (EDA) è il primo sguardo ai dati: quante righe e colonne, che tipi di valori, come sono distribuiti.

In [ ]:
df = pd.read_csv("data/pokemon.csv")  # carichiamo il dataset usando pandas
df.head()

In [ ]:
df.shape # (righe, colonne): quanti esempi e quante feature

In [ ]:
df.info() # nome e tipo di ogni colonna, e quanti valori mancanti

In [ ]:
nomi_pokemon = df["Name"].unique().tolist()  
print("Pokemon nel dataset: ", len(nomi_pokemon))
print("Primi 10 pokemon (in ordine di comparsa nel dataset): ", nomi_pokemon[:10])

In [ ]:
df.loc[df["Name"] == "Bulbasaur"]

In [ ]:
stat_cols = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"] # nomi delle colonne di statistiche che ci interessano
df[stat_cols].describe()   # statistiche di base (media, min, max, quartili) delle colonne numeriche

In [ ]:
df["Attack"].hist(bins=20) # distribuzione di una singola feature
plt.xlabel("Attack");
plt.ylabel("conteggio");
plt.show()

In [ ]:
df.plot.scatter(x="Attack", y="Defense")    # relazione tra due feature
plt.show()

Se la rappresentazione è buona, oggetti simili diventano punti vicini nello spazio dei vettori che li rappresentano.

<div style="text-align: center;"><img src="images/distanza.png" width="400"></div>

Nel nostro esempio: descriviamo ogni Pokémon con le sue 6 statistiche. I vicini di Pikachu sono Pokémon con un profilo simile (poca difesa, discreta velocità): potremmo usarli per suggerire un “sostituto” simile, oppure per scoprire archetipi (tank, veloce-fragile, …) semplicemente guardando chi sta vicino a chi.

Altri esempi:
* _Sistema di raccomandazione_: ti suggerisce gli oggetti (per esempio film o canzoni) vicini a quelli che ti sono piaciuti;
* _Ricerca_: cerchi oggetti simili a un input e ricevi i suoi vicini (ricerche correlate).

Come possiamo misurare la _vicinanza_ tra oggetti?

![distanze](images/euclide_vs_coseno.png)

In [ ]:
def distanza_esteso(a, b):
    diff = (a - b)
    sq_diff = diff ** 2
    sum_sq_diff = 0
    for sq_diff_i in sq_diff:   # itero le righe del dataset
        sum_sq_diff += sq_diff_i
    dist = sum_sq_diff ** (1/2)
    return dist

def distanza(a, b):
    # sfruttando torch / numpy
    return ((a - b)**2).sum().sqrt()

In [ ]:
X = torch.tensor(df[stat_cols].to_numpy(), dtype=torch.float32)  # abbiamo preso le colonne numeriche
print("Dimensione dati: ", X.shape)
print(X[:10])
print("Prima riga dataset (GLI INDICI PARTONO DA 0): ", X[0])

In [ ]:
print("Distanza calcolata con operazioni torch: ", distanza(X[0], X[1]))
print("Distanza calcolata con python 'semplice': ", distanza_esteso(X[0], X[1]))
print("Torch totale: ", torch.cdist(X[0:1], X[1:2])[0])

Troviamo i _vicini_ (ovvero i punti più simili, con la distanza minore) a un dato oggetto:

In [ ]:
# distanza di tutti i punti dalla riga 0
dist_from_0 = torch.cdist(X[0:1], X)
dist_from_0.shape

In [ ]:
# distanza di tutti i punti da una riga generica i
i = 0
dist_from_i = torch.cdist(X[i:i+1], X)
dist_from_i.shape

In [ ]:
dist_from_i.squeeze().shape

In [ ]:
# distanze in ordine crescente
torch.sort(dist_from_i).values.squeeze()[:50]

In [ ]:
# a quale punto (ovvero, a quale vicino) corrispondono?
torch.sort(dist_from_i).indices.squeeze()[:50]

In [ ]:
def vicini_di(i, k=3):
    dists = torch.cdist(X[i:i+1], X).squeeze(0)
    ordine = torch.sort(dists).indices
    # prendiamo i k più vicini
    return ordine[1:k+1]  # la distanza da sè stessi è sempre 0, escludiamo

In [ ]:
# prendiamo i vicini di Pikachu
nomi = df["Name"].tolist()
i = nomi.index("Pikachu")  # in quale riga è Pikachu?
print("Indice della riga in cui si trova Pikachu: ", i)
vicini_i = vicini_di(i, k=3)
for j in vicini_i:
    print(nomi[j])

## Variabili categoriche

Finora abbiamo descritto ogni oggetto con valori numerici (le statistiche dei Pokémon) e su quelli abbiamo misurato distanze. Ma non tutte le caratteristiche sono numeri: molte sono categoriche, cioè prendono un valore in un insieme discreto e finito di possibilità (poche etichette, come un colore o una categoria).

In [ ]:
df["Type 1"].value_counts() # quante volte compare ogni categoria (di una colonna categorica)

Come posso rappresentare numericamente una variabile categorica?

![encoding](images/encoding.png)

In [ ]:
type_1 = df["Type 1"].astype("category")

# 1) Encoding Intero
codici = type_1.cat.codes
print("Encoding intero: ")
codici.head(n=25)

In [ ]:
# 2) One-hot Encoding: niente ordine falso (input onesto e standard), ma sparso
onehot = pd.get_dummies(type_1).astype(int)
print("One-hot encoding: ")
onehot.head()